# 🔌 Managed MCP Connectors — Live Doc Lookup for Compliance 📚

Welcome! Compliance analysts constantly ask "what does the official guidance say?" Instead of pasting docs, we give the agent a **managed Model Context Protocol (MCP) connector** to the **Microsoft Learn MCP server** so it can search authoritative docs at runtime.

In this notebook we'll:

1. **Initialize** an `AIProjectClient` and attach the public **Microsoft Learn MCP** (`https://learn.microsoft.com/api/mcp`) as a hosted tool.
2. **Create a compliance research agent** that grounds answers in live docs.
3. **Ask FSI doc-lookup questions** and watch the MCP tool fetch real content.

### ⚠️ Disclaimer ⚠️
> Educational/demo only — verify all regulatory guidance with official sources.

> 💡 Managed = Foundry runs the MCP calls; auth lives in a connection, not your code.


In [ ]:
# 1. Setup — connect and define the managed Learn MCP tool
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

load_dotenv()
project_endpoint = os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"]
model = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

project_client = AIProjectClient(endpoint=project_endpoint, credential=AzureCliCredential(), allow_preview=True)
openai_client = project_client.get_openai_client()

learn_mcp = MCPTool(
    server_label="microsoft_learn",
    server_url="https://learn.microsoft.com/api/mcp",  # public, anonymous
    require_approval="never",
)
print("✅ Connected:", project_endpoint)
print("🔌 MCP server:", learn_mcp.server_url)


## 2. Create a compliance research agent with the MCP tool 🛡️

The agent attaches the hosted MCP tool. For public servers no connection is needed; for private/enterprise servers add a `project_connection_id` and keep `require_approval="always"` for an approval gate.


In [ ]:
# 2. Create the agent with the Learn MCP tool attached
agent = project_client.agents.create_version(
    agent_name="compliance-doc-research",
    definition=PromptAgentDefinition(
        model=model,
        instructions=(
            "You are a financial-services compliance research assistant. Use the Microsoft Learn "
            "MCP tools to look up official docs, cite page titles/URLs, and keep answers concise."
        ),
        tools=[learn_mcp],
    ),
)
print(f"✅ Agent: {agent.name} v{agent.version} (Learn MCP attached)")


## 3. Ask FSI doc-lookup questions 🔎

Foundry executes the MCP calls server-side and feeds the results back to the model. We print the response plus any MCP tool items so you can see the connector firing.


In [ ]:
# 3. Doc-lookup queries via the managed MCP connector
queries = [
    "Use Microsoft Learn to summarize built-in evaluators available for assessing AI agents in Foundry.",
    "What does the Foundry guidance say about red teaming AI agents for safety? Cite the page.",
]
for q in queries:
    r = openai_client.responses.create(
        input=q,
        extra_body={"agent_reference": {"type": "agent_reference", "name": agent.name}},
    )
    tool_calls = [i.type for i in (r.output or []) if "mcp" in getattr(i, "type", "")]
    print(f"👤 {q}\n🔌 MCP items: {tool_calls or 'none'}")
    print("🤖", (r.output_text or "").strip()[:320], "…\n" + "=" * 60)


In [ ]:
# 4. Cleanup — delete the agent
try:
    # [kept-in-foundry] project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
    print(f"🗑️ Deleted {agent.name}")
except Exception as e:
    print(f"⚠️ Could not delete agent: {e}")
project_client.close()
print("✅ Cleanup complete")

## 📚 References

- [Connect agents to Model Context Protocol servers](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/model-context-protocol) — managed MCP tool + connection auth
- [azure-ai-projects (Python) reference](https://learn.microsoft.com/python/api/overview/azure/ai-projects-readme?view=azure-python) — `MCPTool` and connection IDs
- [What is Microsoft Foundry?](https://learn.microsoft.com/azure/foundry/what-is-foundry) — connectors and namespaces in context
- [Quickstart: Create a prompt agent](https://learn.microsoft.com/azure/foundry/agents/quickstarts/prompt-agent) — base agent the MCP tools attach to
> 💡 Tip: search the official docs live from your agent via the **Microsoft Learn MCP** at `https://learn.microsoft.com/api/mcp`.